# WTI Crude Oil Price Analysis

Download the latest three years of daily West Texas Intermediate (WTI) spot prices from the U.S. Energy Information Administration series published by FRED (`DCOILWTICO`). The notebook calculates price changes, returns, moving averages, annualized volatility, and drawdowns.

FRED observations are daily on business days and may be delayed. Percentage-return analysis assumes prices in the selected window are positive.

In [ ]:
from datetime import date
from urllib.parse import urlencode

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

FRED_CSV_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv"
WTI_SERIES_ID = "DCOILWTICO"
TRADING_DAYS_PER_YEAR = 252

## Data loading

`fetch_wti_data` requests only the requested date range, converts FRED's missing-value markers to nulls, and returns a date-indexed frame with prices in U.S. dollars per barrel.

In [ ]:
def fetch_wti_data(start_date=None, end_date=None):
    """Fetch daily WTI spot prices from FRED for a date range.

    By default, the range is the three years ending today. Returns a DataFrame
    indexed by date with one ``price`` column (USD per barrel).
    """
    end = pd.Timestamp(end_date or date.today()).normalize()
    start = pd.Timestamp(start_date) if start_date else end - pd.DateOffset(years=3)
    start = start.normalize()
    if start > end:
        raise ValueError("start_date must be on or before end_date")

    query = urlencode(
        {
            "id": WTI_SERIES_ID,
            "cosd": start.strftime("%Y-%m-%d"),
            "coed": end.strftime("%Y-%m-%d"),
        }
    )
    raw = pd.read_csv(f"{FRED_CSV_URL}?{query}")
    if WTI_SERIES_ID not in raw.columns or raw.empty:
        raise ValueError(f"FRED returned no {WTI_SERIES_ID} observations for the requested range")

    date_column = raw.columns[0]
    prices = raw.rename(columns={date_column: "date", WTI_SERIES_ID: "price"})[["date", "price"]]
    prices["date"] = pd.to_datetime(prices["date"], errors="raise")
    prices["price"] = pd.to_numeric(prices["price"], errors="coerce")
    prices = prices.dropna(subset=["price"]).drop_duplicates(subset="date", keep="last")
    prices = prices.set_index("date").sort_index()
    prices = prices.loc[(prices.index >= start) & (prices.index <= end)]

    if prices.empty:
        raise ValueError(f"No usable {WTI_SERIES_ID} observations found in the requested date range")
    return prices

## Price and return calculations

The analysis frame includes daily dollar and percentage changes, a normalized price index, 20/50/200-observation moving averages, 20-observation annualized volatility, and price drawdown from the previous peak.

In [ ]:
def analyze_prices(prices):
    """Calculate common technical and risk measures for a WTI price frame."""
    if "price" not in prices.columns:
        raise ValueError("prices must contain a 'price' column")
    if prices.empty:
        raise ValueError("prices must contain at least one observation")

    result = prices[["price"]].copy().sort_index()
    if result["price"].isna().any() or (result["price"] <= 0).any():
        raise ValueError("prices must be non-null and positive for percentage-return analysis")

    result["daily_change"] = result["price"].diff()
    result["daily_return"] = result["price"].pct_change(fill_method=None)
    result["return_index"] = result["price"] / result["price"].iloc[0]
    result["ma_20"] = result["price"].rolling(20, min_periods=20).mean()
    result["ma_50"] = result["price"].rolling(50, min_periods=50).mean()
    result["ma_200"] = result["price"].rolling(200, min_periods=200).mean()
    result["volatility_20d_annualized"] = (
        result["daily_return"].rolling(20, min_periods=20).std()
        * np.sqrt(TRADING_DAYS_PER_YEAR)
    )
    result["drawdown"] = result["price"] / result["price"].cummax() - 1
    return result


def summarize_analysis(analysis):
    """Return key period-level metrics as a labeled Series."""
    if analysis.empty:
        raise ValueError("analysis must contain at least one observation")

    returns = analysis["daily_return"].dropna()
    return pd.Series(
        {
            "Start date": analysis.index.min().date(),
            "End date": analysis.index.max().date(),
            "Observations": len(analysis),
            "Starting price (USD/barrel)": analysis["price"].iloc[0],
            "Latest price (USD/barrel)": analysis["price"].iloc[-1],
            "Cumulative return": analysis["return_index"].iloc[-1] - 1,
            "Annualized daily-return volatility": returns.std() * np.sqrt(TRADING_DAYS_PER_YEAR),
            "Maximum drawdown": analysis["drawdown"].min(),
        },
        name="Value",
    )

## Visualization

The chart shows the spot price with moving averages and the drawdown from the running price peak.

In [ ]:
def plot_analysis(analysis):
    """Plot WTI prices, moving averages, and drawdown."""
    if analysis.empty:
        raise ValueError("analysis must contain at least one observation")

    fig, (price_axis, drawdown_axis) = plt.subplots(
        2, 1, figsize=(13, 8), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
    )
    price_axis.plot(analysis.index, analysis["price"], label="WTI spot price", linewidth=1.5)
    for window in (20, 50, 200):
        price_axis.plot(analysis.index, analysis[f"ma_{window}"], label=f"{window}-observation average")
    price_axis.set_title("WTI Crude Oil Spot Price")
    price_axis.set_ylabel("USD per barrel")
    price_axis.grid(alpha=0.25)
    price_axis.legend()

    drawdown_axis.fill_between(analysis.index, analysis["drawdown"] * 100, 0, alpha=0.35)
    drawdown_axis.set_ylabel("Drawdown (%)")
    drawdown_axis.set_xlabel("Date")
    drawdown_axis.grid(alpha=0.25)
    fig.tight_layout()
    return fig, (price_axis, drawdown_axis)

## Run the analysis

Re-run this section whenever you want the latest available three-year window. The functions above also accept explicit `start_date` and `end_date` values for a custom period.

In [ ]:
wti_prices = fetch_wti_data()
wti_analysis = analyze_prices(wti_prices)

print(f"Loaded {len(wti_prices):,} observations from {wti_prices.index.min().date()} through {wti_prices.index.max().date()}.")
display(summarize_analysis(wti_analysis).to_frame())
plot_analysis(wti_analysis)
plt.show()